# 04. Off-Policy Evaluation (OPE)

## 목표
KuaiRec의 **완전관측 행렬**을 ground-truth로 활용하여, 여러 추천 정책의 성능을 오프라인에서 평가한다.

### 핵심 개념
- **완전관측 행렬**: 1,411 유저 × 3,327 비디오의 거의 모든 조합에 대한 시청 데이터 존재
- 이를 통해 어떤 추천 정책이든 "실제로 추천했으면 어떤 반응이었을지" 계산 가능
- OPE 추정량 (IPS, DR, SNIPS)의 정확도를 ground-truth와 비교 검증

---

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

DATA_DIR = Path('../data')

# KuaiRec 완전관측 행렬 로드
small_mat = pd.read_csv(DATA_DIR / 'kuairec/KuaiRec 2.0/data/small_matrix.csv')
user_feat = pd.read_csv(DATA_DIR / 'kuairec/KuaiRec 2.0/data/user_features.csv')

print(f"Small Matrix: {small_mat.shape[0]:,} rows")
print(f"Users: {small_mat['user_id'].nunique():,} | Videos: {small_mat['video_id'].nunique():,}")
print(f"Columns: {list(small_mat.columns)}")

## 1. 보상 행렬 구축 + 추천 정책 구현

In [ ]:
# 이진 보상: watch_ratio >= 2.0이면 "좋아한다" (반복 시청)
REWARD_THRESHOLD = 2.0
small_mat['reward'] = (small_mat['watch_ratio'] >= REWARD_THRESHOLD).astype(int)

# 유저-아이템 보상 행렬 (pivot)
users = sorted(small_mat['user_id'].unique())
videos = sorted(small_mat['video_id'].unique())

reward_matrix = small_mat.pivot_table(
    index='user_id', columns='video_id', values='reward', aggfunc='max'
).fillna(0)

# watch_ratio 행렬 (연속형)
wr_matrix = small_mat.pivot_table(
    index='user_id', columns='video_id', values='watch_ratio', aggfunc='mean'
).fillna(0)

print(f"Reward Matrix: {reward_matrix.shape}")
print(f"보상 비율 (전체): {reward_matrix.values.mean():.4f}")
print(f"유저별 평균 보상: {reward_matrix.mean(axis=1).mean():.4f}")

In [ ]:
# 4가지 추천 정책 구현
np.random.seed(42)
K = 10  # Top-K 추천

def policy_random(user_id, all_videos, k=K):
    """Random Policy: 랜덤으로 K개 추천"""
    return np.random.choice(all_videos, size=k, replace=False)

def policy_popular(user_id, all_videos, k=K):
    """Popular Policy: 전체 인기도 순으로 K개 추천"""
    return popular_items[:k]

def policy_user_cf(user_id, all_videos, k=K):
    """UserCF Policy: 유저 협업 필터링 기반 K개 추천"""
    if user_id not in user_sim_dict:
        return policy_popular(user_id, all_videos, k)
    # 유사 유저 50명의 선호 아이템
    sim_users = user_sim_dict[user_id]
    scores = np.zeros(len(all_videos))
    for su, sim in sim_users:
        if su in reward_matrix.index:
            scores += sim * reward_matrix.loc[su].values
    top_k = np.argsort(scores)[::-1][:k]
    return np.array(all_videos)[top_k]

def policy_item_pop_blend(user_id, all_videos, k=K):
    """Blend Policy: 유저가 좋아한 아이템과 유사한 인기 아이템"""
    user_liked = reward_matrix.loc[user_id]
    liked_items = user_liked[user_liked > 0].index.tolist()
    if len(liked_items) == 0:
        return policy_popular(user_id, all_videos, k)
    # 좋아한 아이템 50% + 인기 50%
    n_liked = min(k // 2, len(liked_items))
    liked_pick = np.random.choice(liked_items, size=n_liked, replace=False)
    remaining = [v for v in popular_items if v not in liked_pick][:k - n_liked]
    return np.concatenate([liked_pick, remaining])

# 인기 아이템 사전 계산
item_popularity = reward_matrix.sum(axis=0).sort_values(ascending=False)
popular_items = item_popularity.index.values
all_videos_arr = reward_matrix.columns.values

# UserCF 유사도 사전 계산 (코사인 유사도, 상위 50명)
print("Computing user similarity matrix...")
user_sim_matrix = cosine_similarity(reward_matrix.values)
user_sim_dict = {}
user_ids = reward_matrix.index.values
for i, uid in enumerate(user_ids):
    sims = user_sim_matrix[i]
    top_idx = np.argsort(sims)[::-1][1:51]  # 자기 자신 제외, 상위 50
    user_sim_dict[uid] = [(user_ids[j], sims[j]) for j in top_idx]

print("Policies ready!")
print(f"  K = {K} (Top-K recommendations)")
print(f"  Policies: Random, Popular, UserCF, Blend")

## 2. Ground-Truth 성능 평가 (완전관측 행렬 활용)

In [ ]:
# Ground-Truth: 완전관측 행렬이 있으므로 실제 성능 계산 가능
policies = {
    'Random': policy_random,
    'Popular': policy_popular,
    'UserCF': policy_user_cf,
    'Blend': policy_item_pop_blend,
}

gt_results = {}
for name, policy_fn in policies.items():
    precisions = []
    recalls = []
    rewards_sum = []
    
    for uid in user_ids:
        # 정책이 추천하는 K개 아이템
        rec_items = policy_fn(uid, all_videos_arr, K)
        
        # Ground-truth 보상
        user_rewards = reward_matrix.loc[uid]
        rec_rewards = user_rewards[rec_items].values
        
        # 유저의 전체 좋아한 아이템
        total_liked = user_rewards.sum()
        
        # Precision@K, Recall@K
        hits = rec_rewards.sum()
        precisions.append(hits / K)
        recalls.append(hits / max(total_liked, 1))
        rewards_sum.append(rec_rewards.mean())
    
    gt_results[name] = {
        'Precision@K': np.mean(precisions),
        'Recall@K': np.mean(recalls),
        'Avg Reward': np.mean(rewards_sum),
    }

gt_df = pd.DataFrame(gt_results).T
gt_df = gt_df.round(4)
print(f"=== Ground-Truth Policy Performance (K={K}) ===\n")
gt_df

## 3. OPE 추정량 구현 및 비교

로그 데이터(Random Policy로 수집)를 사용하여, 다른 정책의 성능을 **오프라인**으로 추정한다.

In [ ]:
# 로그 데이터 시뮬레이션: Random Policy로 수집된 데이터
# 각 유저에게 랜덤으로 K개 추천 → 보상 기록
np.random.seed(42)
n_logs_per_user = 20  # 유저당 20개 로그

log_data = []
for uid in user_ids:
    shown_items = np.random.choice(all_videos_arr, size=n_logs_per_user, replace=False)
    for item in shown_items:
        reward = reward_matrix.loc[uid, item] if item in reward_matrix.columns else 0
        log_data.append({
            'user_id': uid,
            'video_id': item,
            'reward': reward,
            'propensity': n_logs_per_user / len(all_videos_arr)  # Random policy propensity
        })

log_df = pd.DataFrame(log_data)
print(f"Logged data: {len(log_df):,} rows")
print(f"Logging policy: Random (propensity = {n_logs_per_user/len(all_videos_arr):.6f})")

# OPE 추정량 구현
def ope_dm(log_df, target_policy_fn, reward_model_matrix, user_ids, all_videos, k=K):
    """Direct Method (DM): 보상 모델로 직접 추정"""
    values = []
    for uid in user_ids:
        rec_items = target_policy_fn(uid, all_videos, k)
        predicted_rewards = [reward_model_matrix.loc[uid, item] 
                           if item in reward_model_matrix.columns else 0 
                           for item in rec_items]
        values.append(np.mean(predicted_rewards))
    return np.mean(values)

def ope_ips(log_df, target_policy_fn, user_ids, all_videos, k=K):
    """Inverse Propensity Scoring (IPS)"""
    values = []
    for uid in user_ids:
        user_log = log_df[log_df['user_id'] == uid]
        rec_items = set(target_policy_fn(uid, all_videos, k))
        
        ips_val = 0
        for _, row in user_log.iterrows():
            if row['video_id'] in rec_items:
                # target policy probability (uniform in rec set)
                target_prob = 1.0 / k
                ips_val += row['reward'] * target_prob / row['propensity']
        values.append(ips_val / len(user_log))
    return np.mean(values)

def ope_snips(log_df, target_policy_fn, user_ids, all_videos, k=K):
    """Self-Normalized IPS (SNIPS) — variance reduction"""
    numerator = 0
    denominator = 0
    for uid in user_ids:
        user_log = log_df[log_df['user_id'] == uid]
        rec_items = set(target_policy_fn(uid, all_videos, k))
        
        for _, row in user_log.iterrows():
            if row['video_id'] in rec_items:
                w = (1.0 / k) / row['propensity']
                numerator += w * row['reward']
                denominator += w
    return numerator / max(denominator, 1e-10)

print("OPE estimators ready!")

In [ ]:
# 각 정책에 대해 OPE 추정량 계산 + Ground-Truth 비교
# 성능을 위해 유저 200명 샘플링
np.random.seed(42)
sample_users = np.random.choice(user_ids, size=min(200, len(user_ids)), replace=False)
sample_log = log_df[log_df['user_id'].isin(sample_users)]

ope_results = {}
for policy_name, policy_fn in policies.items():
    if policy_name == 'Random':
        continue  # 로그 수집 정책과 동일하므로 스킵
    
    print(f"Evaluating {policy_name}...")
    
    # Ground Truth (샘플 유저)
    gt_vals = []
    for uid in sample_users:
        rec = policy_fn(uid, all_videos_arr, K)
        gt_vals.append(reward_matrix.loc[uid, rec].mean())
    gt_value = np.mean(gt_vals)
    
    # DM
    dm_value = ope_dm(sample_log, policy_fn, reward_matrix, sample_users, all_videos_arr)
    
    # IPS
    ips_value = ope_ips(sample_log, policy_fn, sample_users, all_videos_arr)
    
    # SNIPS
    snips_value = ope_snips(sample_log, policy_fn, sample_users, all_videos_arr)
    
    ope_results[policy_name] = {
        'Ground Truth': gt_value,
        'DM': dm_value,
        'IPS': ips_value,
        'SNIPS': snips_value,
        'DM Error (%)': abs(dm_value - gt_value) / max(gt_value, 1e-10) * 100,
        'IPS Error (%)': abs(ips_value - gt_value) / max(gt_value, 1e-10) * 100,
        'SNIPS Error (%)': abs(snips_value - gt_value) / max(gt_value, 1e-10) * 100,
    }

ope_df = pd.DataFrame(ope_results).T.round(4)
print("\n=== OPE vs Ground Truth ===\n")
ope_df

In [ ]:
# OPE 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: 추정값 비교
policy_names = list(ope_results.keys())
x = np.arange(len(policy_names))
width = 0.2

for i, est in enumerate(['Ground Truth', 'DM', 'IPS', 'SNIPS']):
    vals = [ope_results[p][est] for p in policy_names]
    colors = ['#2c3e50', '#3498db', '#e74c3c', '#2ecc71']
    axes[0].bar(x + i*width, vals, width, label=est, color=colors[i], alpha=0.8)

axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(policy_names)
axes[0].set_ylabel('Estimated Value')
axes[0].set_title('OPE Estimates vs Ground Truth')
axes[0].legend()

# 오른쪽: 오차율 비교
for i, est in enumerate(['DM Error (%)', 'IPS Error (%)', 'SNIPS Error (%)']):
    vals = [ope_results[p][est] for p in policy_names]
    colors = ['#3498db', '#e74c3c', '#2ecc71']
    axes[1].bar(x + i*width, vals, width, label=est.replace(' Error (%)', ''), color=colors[i], alpha=0.8)

axes[1].set_xticks(x + width)
axes[1].set_xticklabels(policy_names)
axes[1].set_ylabel('Relative Error (%)')
axes[1].set_title('OPE Estimation Error (Lower is Better)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../notebooks/figures/04_ope_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. OPE 결론

### 주요 발견
1. **정책 성능 순위**: UserCF > Blend > Popular > Random — 협업 필터링이 가장 효과적
2. **DM 추정량**: 완전관측 행렬을 보상 모델로 사용 → Ground Truth와 거의 일치 (편향 없음)
3. **IPS 추정량**: 고분산 — 로그 데이터와 타겟 정책 간 겹침이 적을수록 불안정
4. **SNIPS 추정량**: IPS 대비 분산 감소 — 자기정규화로 안정적

### 핵심 인사이트
- KuaiRec의 완전관측 행렬 덕분에 OPE 추정량의 정확도를 **직접 검증** 가능
- 실제 서비스에서는 Ground Truth가 없으므로, OPE 추정량의 신뢰성이 매우 중요
- DM은 보상 모델 정확도에 의존, IPS/SNIPS는 propensity 추정 정확도에 의존

### Next Steps
- `05_filter_bubble.ipynb`: 추천 정책이 콘텐츠 다양성에 미치는 영향